# NumPy + Pandas Mini Project: Smart Store Sales Analysis

In this notebook, I use **NumPy and Pandas together** to build a realistic store-sales analysis.

I generate 90 days of product data, clean missing values, create new features, analyze profit and sales, detect outliers, build pivot tables, calculate rolling averages, and rank products with a custom performance score.

> NumPy helps me create and transform numerical data.  
> Pandas helps me organize, filter, summarize, and interpret it.


## What I Practice

**NumPy:** random data generation, arrays, vectorized operations, `np.where`, `np.select`, percentiles, broadcasting, reproducible results.

**Pandas:** DataFrame creation, cleaning, feature engineering, grouping, filtering, sorting, pivot tables, rolling averages, correlations, and ranking.


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
np.random.seed(42)

## 1. Creating the Product Structure

In [ ]:
products = np.array([
    "Gaming Laptop",
    "Mechanical Keyboard",
    "Wireless Mouse",
    "27-inch Monitor",
    "USB-C Hub",
    "Noise-Cancelling Headphones"
])

categories = np.array([
    "Computers", "Accessories", "Accessories",
    "Displays", "Accessories", "Audio"
])

base_prices = np.array([54999, 3299, 1899, 12499, 1499, 8999])
base_demand = np.array([8, 28, 40, 16, 35, 20])

dates = pd.date_range("2026-01-01", periods=90, freq="D")

store_data = pd.MultiIndex.from_product(
    [dates, products],
    names=["Date", "Product"]
).to_frame(index=False)

category_map = dict(zip(products, categories))
price_map = dict(zip(products, base_prices))
demand_map = dict(zip(products, base_demand))

store_data["Category"] = store_data["Product"].map(category_map)
store_data["Base Price"] = store_data["Product"].map(price_map)
store_data["Base Demand"] = store_data["Product"].map(demand_map)

store_data.head()

## 2. Generating Daily Sales

I add weekend demand, discount effects, random variation, and stock limits.  
The result is more realistic than manually typing a few fixed values.


In [ ]:
row_count = len(store_data)

weekend_multiplier = np.where(
    store_data["Date"].dt.dayofweek.to_numpy() >= 5,
    1.20,
    1.00
)

store_data["Discount Rate"] = np.random.choice(
    [0.00, 0.05, 0.10, 0.15, 0.20],
    size=row_count,
    p=[0.40, 0.20, 0.20, 0.15, 0.05]
)

demand_noise = np.random.normal(1.00, 0.18, row_count)
discount_multiplier = 1 + store_data["Discount Rate"].to_numpy() * 1.80

expected_units = (
    store_data["Base Demand"].to_numpy()
    * weekend_multiplier
    * demand_noise
    * discount_multiplier
)

store_data["Stock Available"] = np.random.randint(5, 65, row_count)

store_data["Units Sold"] = np.minimum(
    np.maximum(np.rint(expected_units), 0),
    store_data["Stock Available"].to_numpy()
).astype(int)

store_data.head()

## 3. Revenue, Cost, Profit, and Margin

In [ ]:
store_data["Sale Price"] = (
    store_data["Base Price"] * (1 - store_data["Discount Rate"])
)

cost_ratio = np.random.uniform(0.58, 0.76, row_count)
store_data["Unit Cost"] = store_data["Base Price"] * cost_ratio

store_data["Revenue"] = (
    store_data["Sale Price"] * store_data["Units Sold"]
)

store_data["Total Cost"] = (
    store_data["Unit Cost"] * store_data["Units Sold"]
)

store_data["Profit"] = store_data["Revenue"] - store_data["Total Cost"]

store_data["Profit Margin (%)"] = np.where(
    store_data["Revenue"] > 0,
    store_data["Profit"] / store_data["Revenue"] * 100,
    0
)

store_data.head()

## 4. Adding and Cleaning Missing Ratings

In [ ]:
ratings = np.clip(
    np.random.normal(4.35, 0.35, row_count),
    1,
    5
)

missing_positions = np.random.choice(
    store_data.index,
    size=25,
    replace=False
)

store_data["Customer Rating"] = ratings
store_data.loc[missing_positions, "Customer Rating"] = np.nan

print("Missing values before cleaning:")
print(store_data.isna().sum())

In [ ]:
store_data["Customer Rating"] = (
    store_data.groupby("Product")["Customer Rating"]
    .transform(lambda values: values.fillna(values.median()))
)

print("Missing ratings after cleaning:",
      store_data["Customer Rating"].isna().sum())

## 5. Feature Engineering

In [ ]:
store_data["Day Type"] = np.where(
    store_data["Date"].dt.dayofweek >= 5,
    "Weekend",
    "Weekday"
)

store_data["Stock Status"] = np.select(
    [
        store_data["Stock Available"] < 15,
        store_data["Stock Available"] < 30
    ],
    ["Critical", "Low"],
    default="Healthy"
)

store_data["Discount Level"] = pd.cut(
    store_data["Discount Rate"],
    bins=[-0.01, 0.00, 0.10, 0.20],
    labels=["No Discount", "Moderate", "High"]
)

store_data["Stock Sell-Through (%)"] = (
    store_data["Units Sold"]
    / store_data["Stock Available"]
    * 100
)

store_data.head()

## 6. Product Performance Summary

In [ ]:
product_summary = (
    store_data.groupby("Product")
    .agg(
        Total_Units_Sold=("Units Sold", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Total_Profit=("Profit", "sum"),
        Average_Profit_Margin=("Profit Margin (%)", "mean"),
        Average_Rating=("Customer Rating", "mean"),
        Average_Discount=("Discount Rate", "mean"),
        Critical_Stock_Days=("Stock Status", lambda x: (x == "Critical").sum())
    )
)

product_summary["Revenue Share (%)"] = (
    product_summary["Total_Revenue"]
    / product_summary["Total_Revenue"].sum()
    * 100
)

product_summary.sort_values("Total_Profit", ascending=False)

## 7. Category Analysis

In [ ]:
category_summary = (
    store_data.groupby("Category")
    .agg(
        Units_Sold=("Units Sold", "sum"),
        Revenue=("Revenue", "sum"),
        Profit=("Profit", "sum"),
        Average_Rating=("Customer Rating", "mean")
    )
    .sort_values("Profit", ascending=False)
)

category_summary

## 8. Weekend vs Weekday Performance

In [ ]:
day_type_summary = (
    store_data.groupby("Day Type")
    .agg(
        Average_Units_Sold=("Units Sold", "mean"),
        Average_Revenue=("Revenue", "mean"),
        Average_Profit=("Profit", "mean")
    )
)

day_type_summary

## 9. Discount Impact

In [ ]:
discount_summary = (
    store_data.groupby("Discount Level", observed=False)
    .agg(
        Average_Units_Sold=("Units Sold", "mean"),
        Average_Revenue=("Revenue", "mean"),
        Average_Profit=("Profit", "mean"),
        Average_Margin=("Profit Margin (%)", "mean")
    )
)

discount_summary

## 10. Detecting Revenue Outliers with NumPy

In [ ]:
q1, q3 = np.percentile(store_data["Revenue"], [25, 75])
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

revenue_outliers = store_data[
    (store_data["Revenue"] < lower_bound)
    | (store_data["Revenue"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(revenue_outliers))

revenue_outliers[
    ["Date", "Product", "Units Sold", "Revenue"]
].sort_values("Revenue", ascending=False).head(10)

## 11. Seven-Day Rolling Sales Average

In [ ]:
store_data = store_data.sort_values(["Product", "Date"])

store_data["7-Day Average Units"] = (
    store_data.groupby("Product")["Units Sold"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

store_data[
    ["Date", "Product", "Units Sold", "7-Day Average Units"]
].tail(12)

## 12. Revenue Pivot Table

In [ ]:
revenue_pivot = pd.pivot_table(
    store_data,
    values="Revenue",
    index="Product",
    columns="Day Type",
    aggfunc="sum",
    fill_value=0
)

revenue_pivot["Weekend Revenue Share (%)"] = (
    revenue_pivot["Weekend"]
    / revenue_pivot[["Weekday", "Weekend"]].sum(axis=1)
    * 100
)

revenue_pivot.sort_values(
    "Weekend Revenue Share (%)",
    ascending=False
)

## 13. Correlation Analysis

In [ ]:
correlation_columns = [
    "Discount Rate",
    "Stock Available",
    "Units Sold",
    "Customer Rating",
    "Revenue",
    "Profit",
    "Profit Margin (%)"
]

store_data[correlation_columns].corr()

## 14. Building a Product Performance Score

In [ ]:
def min_max_normalize(series):
    value_range = series.max() - series.min()
    if value_range == 0:
        return pd.Series(1.0, index=series.index)
    return (series - series.min()) / value_range

normalized_profit = min_max_normalize(
    product_summary["Total_Profit"]
)

normalized_sales = min_max_normalize(
    product_summary["Total_Units_Sold"]
)

normalized_rating = product_summary["Average_Rating"] / 5

stock_reliability = 1 - min_max_normalize(
    product_summary["Critical_Stock_Days"]
)

product_summary["Performance Score"] = (
    normalized_profit * 0.40
    + normalized_sales * 0.25
    + normalized_rating * 0.20
    + stock_reliability * 0.15
) * 100

product_summary["Performance Rank"] = (
    product_summary["Performance Score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

product_summary.sort_values("Performance Rank")

## 15. Products That Need Attention

In [ ]:
attention_mask = (
    (product_summary["Average_Rating"] < 4.2)
    | (product_summary["Critical_Stock_Days"] > 20)
    | (product_summary["Average_Profit_Margin"] < 20)
)

products_needing_attention = product_summary[attention_mask]

products_needing_attention.sort_values("Performance Score")

## 16. Final Analysis Report

In [ ]:
best_selling_product = product_summary["Total_Units_Sold"].idxmax()
most_profitable_product = product_summary["Total_Profit"].idxmax()
top_product = product_summary["Performance Score"].idxmax()
weakest_product = product_summary["Performance Score"].idxmin()

print("SMART STORE SALES REPORT")
print("-" * 35)
print(f"Total revenue: {store_data['Revenue'].sum():,.2f} TRY")
print(f"Total profit: {store_data['Profit'].sum():,.2f} TRY")
print(f"Total units sold: {store_data['Units Sold'].sum():,}")
print(f"Average rating: {store_data['Customer Rating'].mean():.2f}")
print(f"Best-selling product: {best_selling_product}")
print(f"Most profitable product: {most_profitable_product}")
print(f"Best overall performance: {top_product}")
print(f"Weakest overall performance: {weakest_product}")
print(f"Products needing attention: {products_needing_attention.index.tolist()}")

## Mini Challenges

1. Find the three highest-profit dates.
2. Compare each product's weekend and weekday sales.
3. Find products whose average discount is above the store average.
4. Classify profit values as `Low`, `Medium`, or `High`.
5. Find the product with the most stable sales by using standard deviation.
6. Create a monthly revenue summary.
7. Detect rows where all available stock was sold.
8. Rebuild the performance score with different weights.


## My Notes

- NumPy is useful for generating data and performing fast numerical operations.
- Pandas becomes more powerful when I use it together with NumPy.
- Vectorized operations are cleaner than loops for column calculations.
- Grouping turns raw rows into meaningful summaries.
- High sales volume does not always mean high profit.
- Discounts may increase demand while reducing profit margin.
- Missing values and outliers should be checked before trusting results.
- Rolling averages help me understand trends without being distracted by daily noise.
- A custom score can combine different metrics, but the weights should always be explained.


## Key Takeaways

- NumPy handles numerical logic efficiently.
- Pandas organizes numerical results into readable tables.
- `np.where`, `np.select`, percentiles, and broadcasting are useful for feature engineering.
- `groupby`, `agg`, `pivot_table`, `transform`, and `rolling` are powerful Pandas tools.
- Real analysis requires cleaning, feature engineering, summarization, and interpretation.
- This project is a bridge between basic NumPy/Pandas practice and future machine-learning projects.
